# 03 — Fine-tune MobileNetV3-Large (20→17 class)

Fine-tune từ trọng số 20-class đã train, chuyển sang 17-class.

**Runs:** main (3 seeds) + 3 ablation (seed 42) = **6 runs**

| Stage | Epochs | Strategy | LR |
|---|---|---|---|
| Stage 1 | 5 | Freeze backbone, train head only | 1e-3 |
| Stage 2 | 15 | Unfreeze all, differential LR | backbone=1e-5, head=1e-4 |

In [ ]:
from pathlib import Path
import sys

if Path('/content').exists():
    ROOT = Path('/content/woodbench')
else:
    ROOT = Path.cwd().parent.parent.parent

sys.path.insert(0, str(ROOT))
sys.path.insert(0, str(ROOT / "notebooks" / "phase 1" / "final-17class"))
print(f"ROOT: {ROOT}")

In [ ]:
# Colab-only: install dependencies
if Path('/content').exists():
    get_ipython().system(f'pip install -q -r {ROOT}/requirements.txt')

In [ ]:
# ── Configuration ──────────────────────────────────────────
BACKBONE = "mobilenetv3_large"
DATA_ROOT = ROOT / "data" / "task-B-group-split"
OUTPUT_ROOT = ROOT / "data" / "experiments" / "Task-B-17class"
DEVICE = "cuda"

import torch
if not torch.cuda.is_available():
    print("⚠️ CUDA not available, using CPU (will be slow)")
    DEVICE = "cpu"
else:
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")

In [ ]:
# Verify 17-class dataset is ready
assert DATA_ROOT.exists(), f"Data not found: {DATA_ROOT}"

train_classes = sorted([d.name for d in (DATA_ROOT / "train").iterdir() if d.is_dir()])
val_classes = sorted([d.name for d in (DATA_ROOT / "val").iterdir() if d.is_dir()])
test_classes = sorted([d.name for d in (DATA_ROOT / "test").iterdir() if d.is_dir()])

print(f"Train: {len(train_classes)} classes")
print(f"Val:   {len(val_classes)} classes")
print(f"Test:  {len(test_classes)} classes")

assert len(train_classes) == 17, f"Expected 17 train classes, got {len(train_classes)}"
assert len(val_classes) == 17, f"Expected 17 val classes, got {len(val_classes)}"
assert len(test_classes) == 17, f"Expected 17 test classes, got {len(test_classes)}"
print("\n✅ Dataset verified: 17 classes in all splits")

In [ ]:
# Verify all required checkpoints exist
from finetune_17class import EXPERIMENTS

print("Checkpoint verification:")
all_exist = True
for exp_name, cfg in EXPERIMENTS.items():
    for seed in cfg["seeds"]:
        ckpt = ROOT / cfg["ckpt_pattern"].format(backbone=BACKBONE, seed=seed)
        status = "✅" if ckpt.exists() else "❌ MISSING"
        if not ckpt.exists():
            all_exist = False
        print(f"  {exp_name:<35s} seed={seed} {status}")

if not all_exist:
    print("\n⚠️ Some checkpoints missing — those runs will be skipped")
else:
    print("\n✅ All checkpoints found")

In [ ]:
# ── RUN ALL EXPERIMENTS ────────────────────────────────────
# main: 3 seeds (42, 123, 456)
# ablations: seed 42 only
# Total: 6 runs

from finetune_17class import run_all_experiments
import time

print(f"Starting {BACKBONE} fine-tuning...")
print(f"Output: {OUTPUT_ROOT}")

start = time.time()
results = run_all_experiments(
    backbone_name=BACKBONE,
    root=ROOT,
    data_root=DATA_ROOT,
    output_root=OUTPUT_ROOT,
    device=DEVICE,
)
elapsed = time.time() - start
print(f"\n🏁 Total time: {elapsed/60:.1f} minutes")

In [ ]:
# ── RESULTS SUMMARY ────────────────────────────────────────
import pandas as pd

if results:
    df = pd.DataFrame([
        {
            "Experiment": r["experiment"],
            "Seed": r["seed"],
            "Val Acc (%)": f"{r['best_val_acc']:.2f}",
            "Test Acc (%)": f"{r['test_accuracy']*100:.2f}",
            "Test F1 (%)": f"{r['test_f1_macro']*100:.2f}",
            "Test Prec (%)": f"{r['test_precision_macro']*100:.2f}",
            "Test Recall (%)": f"{r['test_recall_macro']*100:.2f}",
        }
        for r in results
    ])
    print(f"\n{BACKBONE} — 17-class Results")
    print("=" * 80)
    print(df.to_string(index=False))

    # Main config: mean ± std across 3 seeds
    main_results = [r for r in results if r["experiment"] == "main"]
    if len(main_results) > 1:
        import numpy as np
        accs = [r["test_accuracy"]*100 for r in main_results]
        f1s = [r["test_f1_macro"]*100 for r in main_results]
        print(f"\nMain (3 seeds): Acc = {np.mean(accs):.2f} ± {np.std(accs):.2f}%")
        print(f"                F1  = {np.mean(f1s):.2f} ± {np.std(f1s):.2f}%")
else:
    print("No results (all checkpoints missing?)")

In [ ]:
# ── PER-CLASS F1 (Main/seed_42) ────────────────────────────
main_42 = next((r for r in results if r["experiment"] == "main" and r["seed"] == 42), None)

if main_42 and "per_class" in main_42:
    print(f"\n{BACKBONE} — Per-class F1 (main/seed_42)")
    print("─" * 50)
    per_class = main_42["per_class"]
    for cls_name, scores in sorted(per_class.items()):
        f1 = scores["f1"] * 100
        bar = "█" * int(f1 / 2)
        print(f"  {cls_name:<40s} {f1:>6.2f}% {bar}")
else:
    print("No per-class metrics available")